# 39 · Tabela master: uma linha por usuário

Junta, numa única tabela, perfil, entrada, saída, uso do app e comportamento de cada conta.

Decisões de desenho (validadas com a Fernanda em 14/11/2026):
1. **Quem entra**: as 63.929 contas de `usuarios_limpa` (sem contas de teste e sem cópias). Colunas de
   assinatura ficam vazias para quem nunca assinou.
2. **Data de referência do comportamento** (`data_referencia`): o corte (30/09/2026) para quem está
   ativo ou nunca assinou; a véspera da saída para quem saiu.
3. **Objetivo**: texto do onboarding classificado por palavras-chave (aproximado).

Limites:
- treinos só existem a partir de 01/01/2026 e posts a partir de 01/07/2025. As colunas de uso valem
  para a parte da janela de 90 dias que tem dado (`dias_historico_treinos`, `dias_historico_posts`) e
  ficam vazias quando não há nenhum dia de histórico;
- `nivel_engajamento` precisa de 45 dias de histórico; `treinos_30d` de 30; `treinos_7d` de 7;
- `sistema` e `relogio` são a situação na exportação, sem data.

Saídas: `dados/processados/tabela_master.parquet`, `saidas/tabela_master.csv` e
`saidas/tabela_master_dicionario.csv`.

In [1]:
import sys, re, unicodedata, difflib
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
INI_TREINOS = pd.Timestamp("2026-01-01"); INI_POSTS = pd.Timestamp("2025-07-01")
TIPOS = ["corrida", "funcional", "caminhada", "forca", "yoga", "pedal"]

usuarios = pd.read_parquet(P / "usuarios_limpa.parquet")
assin = pd.read_parquet(P / "assinaturas_limpa.parquet")
resumo = pd.read_parquet(P / "assinaturas_resumo_eventos.parquet", columns=["user_id", "ultimo_desligamento", "renovacao_ligada_no_corte"])
trials = pd.read_parquet(P / "trials.parquet", columns=["user_id", "trial_inicio", "desfecho"])
onb = pd.read_parquet(P / "onboarding_limpa.parquet")
pagr = pd.read_parquet(P / "pagamentos_resumo.parquet", columns=["user_id", "n_aprovadas", "receita_liquida", "teve_cupom", "teve_recusa", "teve_devolucao"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "origem", "duracao_min", "treino_com", "gps_ok"])
posts = pd.read_parquet(P / "feed_posts_limpa.parquet", columns=["user_id", "data", "texto"])
amiz = pd.read_parquet(P / "amizades_limpa.parquet")
tickets = pd.read_parquet(P / "tickets_suporte_limpa.parquet", columns=["user_id", "aberto_em", "sobre_falha_gps"])
pesq = pd.read_parquet(P / "pesquisa_saida_limpa.parquet")
camp = pd.read_parquet(P / "campanha_limpa.parquet", columns=["user_id", "grupo"])
for n, d in [("usuarios", usuarios), ("assinaturas", assin), ("trials", trials), ("onboarding", onb), ("pagamentos_resumo", pagr), ("campanha", camp)]:
    print(f"{n:18s} {len(d):>7,} linhas | {d.user_id.nunique():>7,} contas")

usuarios            63,929 linhas |  63,929 contas
assinaturas         32,583 linhas |  32,583 contas
trials              46,996 linhas |  46,996 contas
onboarding          63,929 linhas |  63,929 contas
pagamentos_resumo   32,583 linhas |  32,583 contas
campanha            14,120 linhas |  14,120 contas


## 1. Base: perfil, entrada e saída

In [2]:
m = usuarios[["user_id", "sexo", "data_nascimento", "idade_no_corte", "cidade", "uf", "sistema", "data_cadastro", "cadastro_antes_do_log", "canal_aquisicao", "relogio_conectado"]].copy()
n0 = len(m)
def junta(m, d, nome):
    r = m.merge(d, on="user_id", how="left", validate="one_to_one")
    print(f"+ {nome}: {len(m)} -> {len(r)} linhas")
    assert len(r) == n0
    return r

m["sexo"] = m.sexo.fillna("não informado")
m["ano_nascimento"] = m.data_nascimento.dt.year.astype("Int64")
m["faixa_idade"] = pd.cut(m.idade_no_corte.astype(float), [0, 24, 34, 44, 54, 120], labels=["até 24", "25-34", "35-44", "45-54", "55+"]).astype(object).fillna("não informado")
m = m.rename(columns={"relogio_conectado": "relogio"}).drop(columns=["data_nascimento"])

m = junta(m, onb[["user_id", "influenciador", "objetivo_texto", "nivel", "dias_por_semana"]].rename(columns={"nivel": "nivel_declarado", "dias_por_semana": "dias_por_semana_declarado"}), "onboarding")
m = junta(m, trials.rename(columns={"trial_inicio": "data_trial", "desfecho": "desfecho_trial"}), "trials")
m["fez_trial"] = m.data_trial.notna()
m["data_trial"] = m.data_trial.dt.normalize()
a = assin[["user_id", "inicio", "plano_atual", "canal_cobranca", "ativa_no_corte", "data_fim", "motivo_fim", "duracao_dias", "renovacao_automatica", "inicio_antes_do_log"]].rename(columns={"inicio": "data_inicio_assinatura", "plano_atual": "plano", "inicio_antes_do_log": "assinatura_antes_do_log"})
m = junta(m, a, "assinaturas")
m = junta(m, resumo, "resumo de eventos")
m = junta(m, pagr.rename(columns={"n_aprovadas": "n_cobrancas_aprovadas", "teve_cupom": "entrou_com_cupom"}), "pagamentos")
m = junta(m, camp.rename(columns={"grupo": "grupo_campanha"}), "campanha")
m = junta(m, pesq.loc[pesq.resposta_mais_recente, ["user_id", "motivo_ajustado"]].rename(columns={"motivo_ajustado": "motivo_pesquisa"}), "pesquisa de saída")

m["ja_assinou"] = m.plano.notna()
m["ativa_no_corte"] = m.ativa_no_corte.fillna(False).astype(bool)
m["situacao"] = np.select([m.ativa_no_corte, m.ja_assinou, m.desfecho_trial == "em_andamento", m.desfecho_trial == "expirou"],
                          ["assinante ativo", "ex-assinante", "em teste grátis", "teste expirou, não assinou"], "nunca iniciou teste")
m["tipo_saida"] = m.motivo_fim.map({"renovacao_desligada": "voluntária", "reembolso": "voluntária", "falha_pagamento": "involuntária", "chargeback": "involuntária"})
m["meses_de_casa"] = (m.duracao_dias / 30.4375).round(1)
m["renovacao_desligada"] = np.where(~m.ja_assinou, None, np.where(m.ativa_no_corte, m.renovacao_automatica == "nao", m.motivo_fim == "renovacao_desligada"))
m["renovacao_desligada"] = m.renovacao_desligada.astype("boolean")
m["data_desligamento"] = m.ultimo_desligamento.dt.normalize()
m["data_referencia"] = np.where(m.ja_assinou & ~m.ativa_no_corte, m.data_fim - pd.Timedelta(days=1), CORTE)
m["data_referencia"] = pd.to_datetime(m.data_referencia)
m = m.drop(columns=["renovacao_automatica", "ultimo_desligamento", "renovacao_ligada_no_corte"])
print(); print(m.situacao.value_counts().to_frame("contas"))
print("data_referencia: corte =", int((m.data_referencia == CORTE).sum()), "| véspera da saída =", int((m.data_referencia < CORTE).sum()), "| antes de 2026 =", int((m.data_referencia < INI_TREINOS).sum()))
ref = m.set_index("user_id").data_referencia

+ onboarding: 63929 -> 63929 linhas
+ trials: 63929 -> 63929 linhas


+ assinaturas: 63929 -> 63929 linhas
+ resumo de eventos: 63929 -> 63929 linhas


+ pagamentos: 63929 -> 63929 linhas
+ campanha: 63929 -> 63929 linhas
+ pesquisa de saída: 63929 -> 63929 linhas



                            contas
situacao                          
teste expirou, não assinou   19143
ex-assinante                 17830
assinante ativo              14753
nunca iniciou teste          11733
em teste grátis                470
data_referencia: corte = 46099 | véspera da saída = 17830 | antes de 2026 = 9459


C:\Users\Inteli\AppData\Local\Temp\ipykernel_18360\2915272939.py:26: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  m["ativa_no_corte"] = m.ativa_no_corte.fillna(False).astype(bool)


## 2. Objetivo do onboarding por palavras-chave

In [3]:
def norm(s):
    return unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
CAT = {  # ordem = prioridade
    "bem-estar mental": ["mental", "ansiedade", "dormir", "momento so meu", "celular", "cabeca", "desligar", "desestress"],
    "saúde": ["colesterol", "pressao", "medico", "sedentari", "dor ", "costas", "disposicao", "healthy", "saude", "mexer", "feel better"],
    "peso e forma": ["peso", "emagrecer", "secar", "barriga", "definir", "corpo", "em forma", "espelho", "kg", "lose weight", "get fit"],
    "companhia": ["galera", "grupo", "junto", "companhia", "namorad", "amig", "pessoal do trabalho", "treinar com o", "treinar com a"],
    "performance": ["pace", "meu tempo", "evoluir", "nivel", "planilha", "rapido", "melhorar"],
}
VOCAB = {w.strip(): c for c, ws in CAT.items() for w in ws if " " not in w.strip()}
PROVA = r"maratona|marathon|prova|granfondo|triat|desafio|\b\d{1,3}\s?k(m)?\b"
PESSOAL = r"casamento|viagem|festa|formatura|verao|reveillon|aniversario|ferias|praia|apresentacao|ensaio"
PISTA = r"preciso estar|tenho |daqui|semanas|meses|mes que vem|\d{1,2}/\d{1,2}|\d{1,2} de |janeiro|fevereiro|marco|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro|meados|fim de|comeco de| e em | em \d|half marathon"

def classifica(t):
    if not isinstance(t, str) or not t.strip(): return (None, None)
    s = norm(t)
    cat = next((c for c, ws in CAT.items() if any(w in s for w in ws)), None)
    if cat is None:  # erro de digitação: palavra mais parecida do vocabulário
        for tok in re.findall(r"[a-z]+", s):
            p = difflib.get_close_matches(tok, VOCAB, n=1, cutoff=0.8)
            if p: cat = VOCAB[p[0]]; break
    pista = re.search(PISTA, s) is not None
    ev = "prova esportiva" if (re.search(PROVA, s) and pista) else ("evento pessoal" if (re.search(PESSOAL, s) and pista) else "sem evento")
    if cat is None: cat = "só o evento" if ev != "sem evento" else "outro"
    return (cat, ev)
r = m.objetivo_texto.map(classifica)
m["objetivo_categoria"] = r.str[0]; m["objetivo_evento"] = r.str[1]
m["tem_objetivo"] = m.objetivo_texto.notna()
m["objetivo_com_data"] = m.objetivo_evento.isin(["prova esportiva", "evento pessoal"]).where(m.tem_objetivo).astype("boolean")
print("com objetivo:", int(m.tem_objetivo.sum()), "de", len(m))
print(pd.crosstab(m.objetivo_categoria, m.objetivo_evento, margins=True)); print()
print("exemplos por categoria")
for c in m.objetivo_categoria.dropna().unique():
    print(f"-- {c}:", " | ".join(m.loc[m.objetivo_categoria == c, "objetivo_texto"].value_counts().head(3).index))
print("-- 'outro' mais comuns:", m.loc[m.objetivo_categoria == "outro", "objetivo_texto"].value_counts().head(8).to_dict())
for e in ["prova esportiva", "evento pessoal"]:
    print(f"-- {e}:", " | ".join(m.loc[m.objetivo_evento == e, "objetivo_texto"].sample(4, random_state=3)))

com objetivo: 44694 de 63929
objetivo_evento     evento pessoal  prova esportiva  sem evento    All
objetivo_categoria                                                    
bem-estar mental               291               57        5024   5372
companhia                      611              181        6864   7656
outro                            0                0         455    455
performance                    283             2919        3646   6848
peso e forma                  3223             1132        7210  11565
saúde                          569              141        9235   9945
só o evento                   1358             1495           0   2853
All                           6335             5925       32434  44694

exemplos por categoria
-- companhia: achar uma galera pra correr | treinar junto com meu namorado | fazer parte de um grupo
-- peso e forma: perder peso | secar a barriga | ficar em forma de novo
-- bem-estar mental: dormir melhor | ter um momento só meu | sai

## 3. Uso do app na janela de 90 dias até a data de referência

In [4]:
tem = ref[ref >= INI_TREINOS]
print("contas com data de referência a partir de 01/01/2026:", len(tem), "| antes (sem treinos no arquivo):", int((ref < INI_TREINOS).sum()))
a = ativ[ativ.user_id.isin(tem.index)].copy()
a["ref"] = a.user_id.map(ref)
a["dias"] = (a.ref - a.data).dt.days
print("treinos:", len(ativ), "-> das contas com referência em 2026:", len(a), "-> até a data de referência:", int((a.dias >= 0).sum()), "-> nos 90 dias:", int(a.dias.between(0, 89).sum()))
ult = a[a.dias >= 0].groupby("user_id").data.max().rename("ultimo_treino")
w = a[a.dias.between(0, 89)].copy()
w["relogio_"] = w.origem != "app"; w["acomp"] = w.treino_com.notna()
g = w.groupby("user_id").agg(treinos_90d=("data", "size"), dias_ativos_90d=("data", "nunique"), minutos_90d=("duracao_min", "sum"), treinos_relogio=("relogio_", "sum"), treinos_acompanhados_90d=("acomp", "sum"))
g["pct_treinos_relogio"] = (g.treinos_relogio / g.treinos_90d * 100).round(1)
pt = w.groupby(["user_id", "tipo"]).agg(n=("data", "size"), minutos=("duracao_min", "sum")).reset_index().sort_values(["user_id", "n", "minutos"], ascending=[True, False, False])
top = pt.drop_duplicates("user_id").set_index("user_id")
g["tipo_principal"] = top.tipo; g["peso_tipo_principal"] = (top.n / g.treinos_90d * 100).round(1); g["n_tipos"] = pt.groupby("user_id").tipo.nunique()
g = g.join(pt.pivot(index="user_id", columns="tipo", values="n").reindex(columns=TIPOS).add_prefix("treinos_").fillna(0).astype(int))
m = m.join(g.drop(columns=["treinos_relogio"]), on="user_id").join(ult, on="user_id")

hist = ((m.data_referencia - INI_TREINOS).dt.days + 1).clip(lower=0, upper=90)
m["dias_historico_treinos"] = hist
com = hist > 0
zerar = ["treinos_90d", "dias_ativos_90d", "minutos_90d", "treinos_acompanhados_90d", "n_tipos"] + ["treinos_" + t for t in TIPOS]
for c in zerar: m[c] = m[c].where(~com, m[c].fillna(0)).where(com)
m["tipo_principal"] = m.tipo_principal.where(~com, m.tipo_principal.fillna("sem treino")).where(com)
m["ambiente_principal"] = m.tipo_principal.map({"corrida": "ar livre", "caminhada": "ar livre", "pedal": "ar livre", "funcional": "indoor", "forca": "indoor", "yoga": "indoor", "sem treino": "sem treino"})
m["minutos_90d"] = m.minutos_90d.round(0)
m["relogio_sincronizou_90d"] = (m.pct_treinos_relogio.fillna(0) > 0).where(com).astype("boolean")
print("com histórico de treinos:", int(com.sum()), "| janela de 90 dias completa:", int((hist == 90).sum()), "| com treino na janela:", int((m.treinos_90d > 0).sum()))
print(m.loc[com, "tipo_principal"].value_counts().to_dict())

contas com data de referência a partir de 01/01/2026: 54470 | antes (sem treinos no arquivo): 9459


treinos: 1252488 -> das contas com referência em 2026: 1252070 -> até a data de referência: 1248760 -> nos 90 dias: 528545


com histórico de treinos: 54470 | janela de 90 dias completa: 51684 | com treino na janela: 27807
{'sem treino': 26663, 'corrida': 10669, 'funcional': 5916, 'caminhada': 3952, 'forca': 3694, 'pedal': 2699, 'yoga': 877}


In [5]:
# Engajamento na data de referência (definição do grupo, src/engajamento.py)
ini = m.set_index("user_id").data_inicio_assinatura.fillna(m.set_index("user_id").data_cadastro)
e = niveis_engajamento(ativ, tem.index, tem, inicio=ini)
m = junta(m, e[["user_id", "nivel", "treinos_7d", "treinos_30d", "dias_sem_treinar", "nunca_treinou"]].rename(columns={"nivel": "nivel_engajamento", "nunca_treinou": "sem_treino_ate_referencia"}), "engajamento")
h = m.dias_historico_treinos
m["nivel_engajamento"] = m.nivel_engajamento.str[3:].where(h >= 45)
m["treinos_7d"] = m.treinos_7d.where(h >= 7); m["treinos_30d"] = m.treinos_30d.where(h >= 30)
# sem nenhum treino no arquivo: o último treino é desconhecido (os treinos começam em 01/01/2026)
m["dias_sem_treinar"] = m.dias_sem_treinar.where(m.ultimo_treino.notna())
m["sem_treino_ate_referencia"] = m.sem_treino_ate_referencia.astype("boolean")
print("nível de engajamento por situação"); print(pd.crosstab(m.situacao, m.nivel_engajamento.fillna("(sem histórico suficiente)"), margins=True))
sem = h / 7
m["treinos_por_semana"] = (m.treinos_90d / sem).where(h >= 28).round(2)
m["aderencia_meta"] = (m.treinos_por_semana / m.dias_por_semana_declarado.astype(float)).round(2)
print(); print("aderência à meta do onboarding (treinos por semana ÷ dias declarados):", m.aderencia_meta.describe()[["count", "mean", "25%", "50%", "75%"]].round(2).to_dict())

+ engajamento: 63929 -> 63929 linhas
nível de engajamento por situação
nivel_engajamento           (sem histórico suficiente)   alto  baixo  médio  possível churn    All
situacao                                                                                          
assinante ativo                                      0  10689    365   2371            1328  14753
em teste grátis                                      0    331      0    139               0    470
ex-assinante                                     10853   2130    771   3184             892  17830
nunca iniciou teste                                  0    523    307    920            9983  11733
teste expirou, não assinou                           0    742    566   1385           16450  19143
All                                              10853  14415   2009   7999           28653  63929

aderência à meta do onboarding (treinos por semana ÷ dias declarados): {'count': 48278.0, 'mean': 0.23, '25%': 0.0, '50%': 0.02, '75%': 

In [6]:
# Feed: posts nos 90 dias até a referência e os dois temas que antecedem a saída
tx = posts.texto.fillna("").str.lower()
DESP = r"cancelei|não renovei|nao renovei|último mês aqui|ultimo mes aqui|dar um tempo do app|pausa nos treinos e no app|depois eu volto"
OBJ = r"conclu[ií]d|completei|terminei|feita ✅|feito ✅|missão cumprida|missao cumprida|valeu cada treino|agora é descansar|agora e descansar|medalha no peito"
posts["tema"] = np.where(tx.str.contains(DESP), "despedida", np.where(tx.str.contains(OBJ), "objetivo", "treino"))
p = posts[posts.user_id.isin(ref.index)].copy()
p["dias"] = (p.user_id.map(ref) - p.data).dt.days
p = p[p.dias.between(0, 89)]
print("posts:", len(posts), "-> nos 90 dias até a referência de cada conta:", len(p))
gp = p.groupby("user_id").agg(posts_90d=("data", "size"), postou_objetivo_cumprido=("tema", lambda s: (s == "objetivo").any()), postou_despedida=("tema", lambda s: (s == "despedida").any()))
m = m.join(gp, on="user_id")
hp = ((m.data_referencia - INI_POSTS).dt.days + 1).clip(lower=0, upper=90)
m["dias_historico_posts"] = hp
m["posts_90d"] = m.posts_90d.where(hp == 0, m.posts_90d.fillna(0)).where(hp > 0)
for c in ["postou_objetivo_cumprido", "postou_despedida"]:
    m[c] = m[c].where(hp == 0, m[c].fillna(False)).where(hp > 0).astype("boolean")
print("com histórico de posts:", int((hp > 0).sum()), "| postou em 90d:", int((m.posts_90d > 0).sum()), "| objetivo cumprido:", int(m.postou_objetivo_cumprido.sum()), "| despedida:", int(m.postou_despedida.sum()))

# Rede na data de referência (amizades_limpa já está cortada em 30/09/2026)
am = amiz.merge(amiz.rename(columns={"seguidor_id": "seguido_id", "seguido_id": "seguidor_id", "desde": "desde_volta"})[["seguidor_id", "seguido_id", "desde_volta"]], on=["seguidor_id", "seguido_id"], how="left")
print("relações:", len(amiz), "-> com a volta anexada:", len(am))
s1 = am.assign(ref=am.seguidor_id.map(ref)); s1 = s1[s1.desde <= s1.ref]
s1["mutua"] = s1.desde_volta.notna() & (s1.desde_volta <= s1.ref)
s2 = am.assign(ref=am.seguido_id.map(ref)); s2 = s2[s2.desde <= s2.ref]
m = m.join(s1.groupby("seguidor_id").agg(n_seguindo=("seguido_id", "size"), n_mutuas=("mutua", "sum")), on="user_id").join(s2.groupby("seguido_id").size().rename("n_seguidores"), on="user_id")
for c in ["n_seguindo", "n_mutuas", "n_seguidores"]: m[c] = m[c].fillna(0).astype(int)

# Suporte e falha de GPS
t = tickets.assign(ref=tickets.user_id.map(ref)); t = t[t.aberto_em.dt.normalize() <= t.ref]
m = m.join(t.groupby("user_id").agg(n_chamados=("aberto_em", "size"), chamado_falha_gps=("sobre_falha_gps", "any")), on="user_id")
m["n_chamados"] = m.n_chamados.fillna(0).astype(int); m["chamado_falha_gps"] = m.chamado_falha_gps.fillna(False).astype(bool)
m["atingida_falha_gps"] = m.user_id.isin(set(ativ.loc[~ativ.gps_ok, "user_id"]) | set(tickets.loc[tickets.sobre_falha_gps, "user_id"]))
print("chamados até a referência:", len(t), "de", len(tickets), "| contas atingidas pela falha de GPS:", int(m.atingida_falha_gps.sum()))
assert len(m) == n0 and m.user_id.is_unique

posts: 119187 -> nos 90 dias até a referência de cada conta: 41127


C:\Users\Inteli\AppData\Local\Temp\ipykernel_18360\4202336912.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  m[c] = m[c].where(hp == 0, m[c].fillna(False)).where(hp > 0).astype("boolean")
C:\Users\Inteli\AppData\Local\Temp\ipykernel_18360\4202336912.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  m[c] = m[c].where(hp == 0, m[c].fillna(False)).where(hp > 0).astype("boolean")


com histórico de posts: 58828 | postou em 90d: 20524 | objetivo cumprido: 4466 | despedida: 1001
relações: 113207 -> com a volta anexada: 113207


chamados até a referência: 4374 de 4422 | contas atingidas pela falha de GPS: 4508


C:\Users\Inteli\AppData\Local\Temp\ipykernel_18360\4202336912.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  m["n_chamados"] = m.n_chamados.fillna(0).astype(int); m["chamado_falha_gps"] = m.chamado_falha_gps.fillna(False).astype(bool)


## 4. Conferências

In [7]:
print("linhas:", len(m), "| contas distintas:", m.user_id.nunique(), "| colunas:", m.shape[1])
print("assinantes ativos:", int(m.ativa_no_corte.sum()), "(esperado 14.753) | já assinou:", int(m.ja_assinou.sum()), "(esperado 32.583)")
eng0 = pd.read_parquet(P / "engajamento_corte.parquet")
x = m[m.ativa_no_corte].merge(eng0[["user_id", "nivel"]], on="user_id")
print("nível dos ativos igual ao de engajamento_corte:", int((x.nivel.str[3:] == x.nivel_engajamento).sum()), "de", len(x))
r0 = pd.read_parquet(P / "rede_por_conta.parquet")
x = m[m.data_referencia == CORTE].merge(r0, on="user_id", suffixes=("", "_ref"))
print("rede no corte igual a rede_por_conta:", {c: int((x[c] == x[c + "_ref"]).sum()) for c in ["n_seguidores", "n_seguindo", "n_mutuas"]}, "de", len(x))
print("soma dos treinos por tipo = treinos_90d:", bool((m[["treinos_" + t for t in TIPOS]].sum(axis=1)[m.treinos_90d.notna()] == m.treinos_90d.dropna()).all()))
print("ex-assinantes com saída antes de 2026 (uso vazio):", int((m.situacao.eq("ex-assinante") & (m.data_referencia < INI_TREINOS)).sum()))
print("nenhuma data depois do corte:", {c: bool((m[c].dropna() <= CORTE).all()) for c in ["data_cadastro", "data_trial", "data_inicio_assinatura", "data_fim", "data_desligamento", "data_referencia", "ultimo_treino"]})

linhas: 63929 | contas distintas: 63929 | colunas: 80
assinantes ativos: 14753 (esperado 14.753) | já assinou: 32583 (esperado 32.583)
nível dos ativos igual ao de engajamento_corte: 14753 de 14753
rede no corte igual a rede_por_conta: {'n_seguidores': 46099, 'n_seguindo': 46099, 'n_mutuas': 46099} de 46099
soma dos treinos por tipo = treinos_90d: True
ex-assinantes com saída antes de 2026 (uso vazio): 9459
nenhuma data depois do corte: {'data_cadastro': True, 'data_trial': True, 'data_inicio_assinatura': True, 'data_fim': True, 'data_desligamento': True, 'data_referencia': True, 'ultimo_treino': True}


## 5. Ordem das colunas, dicionário e gravação

In [8]:
DIC = [
 ("user_id", "identificação", "Chave da conta", "usuarios"),
 ("situacao", "identificação", "Situação no corte: assinante ativo, ex-assinante, em teste grátis, teste expirou sem assinar, nunca iniciou teste", "assinaturas + trials"),
 ("data_referencia", "identificação", "Data em que uso e comportamento foram medidos: 30/09/2026 para ativos e para quem nunca assinou; véspera da saída para ex-assinantes", "derivada"),
 ("sexo", "perfil", "F, M ou não informado", "usuarios"),
 ("ano_nascimento", "perfil", "Ano de nascimento (vazio se não informado ou inválido)", "usuarios"),
 ("idade_no_corte", "perfil", "Idade em 30/09/2026", "usuarios"),
 ("faixa_idade", "perfil", "até 24, 25-34, 35-44, 45-54, 55+, não informado", "usuarios"),
 ("cidade", "perfil", "Cidade padronizada", "usuarios"),
 ("uf", "perfil", "Estado", "usuarios"),
 ("sistema", "perfil", "android ou ios. Situação na exportação, sem data", "usuarios"),
 ("data_cadastro", "entrada", "Data de cadastro (horário de Brasília)", "usuarios"),
 ("cadastro_antes_do_log", "entrada", "Cadastro antes de 07/2024: nessas coortes quase só há contas que assinaram", "usuarios"),
 ("canal_aquisicao", "entrada", "Canal por onde a conta chegou", "usuarios"),
 ("influenciador", "entrada", "Influenciador citado no onboarding, quando houver", "onboarding"),
 ("fez_trial", "entrada", "Iniciou teste grátis (o log começa em 07/2024)", "trials"),
 ("data_trial", "entrada", "Data de início do teste grátis", "trials"),
 ("desfecho_trial", "entrada", "converteu, expirou ou em_andamento", "trials"),
 ("ja_assinou", "entrada", "Tem assinatura (cada conta tem no máximo uma)", "assinaturas"),
 ("data_inicio_assinatura", "entrada", "Início da assinatura", "assinaturas"),
 ("assinatura_antes_do_log", "entrada", "Assinatura iniciada antes de 07/2024 (só aparecem as que ainda existiam nessa data)", "assinaturas"),
 ("plano", "entrada", "mensal ou anual", "assinaturas"),
 ("canal_cobranca", "entrada", "app_store, google_play, web_cartao ou web_pix", "assinaturas"),
 ("entrou_com_cupom", "entrada", "Primeira cobrança com cupom", "pagamentos"),
 ("ativa_no_corte", "saída", "Assinatura ativa em 30/09/2026", "assinaturas"),
 ("data_fim", "saída", "Fim da assinatura (vazio se ativa ou se nunca assinou)", "assinaturas"),
 ("motivo_fim", "saída", "renovacao_desligada, falha_pagamento, chargeback ou reembolso", "assinaturas"),
 ("tipo_saida", "saída", "voluntária (renovação desligada, reembolso) ou involuntária (falha de pagamento, chargeback)", "derivada"),
 ("duracao_dias", "saída", "Dias de assinatura até a saída ou até o corte", "assinaturas"),
 ("meses_de_casa", "saída", "duracao_dias em meses", "derivada"),
 ("renovacao_desligada", "saída", "Ativos: renovação automática desligada no corte. Ex-assinantes: saíram por renovação desligada", "assinaturas"),
 ("data_desligamento", "saída", "Data do último desligamento da renovação automática", "eventos_assinatura"),
 ("motivo_pesquisa", "saída", "Motivo declarado na pesquisa de saída, ajustado pelo comentário. Só 641 contas que cancelaram pelo site de jun a set/2026", "pesquisa_saida"),
 ("dias_historico_treinos", "uso", "Dias da janela de 90 dias com dados de treino (os treinos começam em 01/01/2026). 0 = colunas de treino vazias", "derivada"),
 ("tipo_principal", "uso", "Modalidade mais frequente na janela (empate: mais minutos); 'sem treino' se não treinou", "atividades"),
 ("ambiente_principal", "uso", "ar livre (corrida, caminhada, pedal), indoor (funcional, força, yoga) ou sem treino", "derivada"),
 ("n_tipos", "uso", "Número de modalidades praticadas na janela", "atividades"),
 ("peso_tipo_principal", "uso", "% dos treinos da janela que são da modalidade principal", "atividades"),
 ("treinos_corrida", "uso", "Treinos de corrida na janela", "atividades"),
 ("treinos_funcional", "uso", "Treinos de funcional na janela", "atividades"),
 ("treinos_caminhada", "uso", "Treinos de caminhada na janela", "atividades"),
 ("treinos_forca", "uso", "Treinos de força na janela", "atividades"),
 ("treinos_yoga", "uso", "Treinos de yoga na janela", "atividades"),
 ("treinos_pedal", "uso", "Treinos de pedal na janela", "atividades"),
 ("relogio", "uso", "nenhum, garmin ou apple_watch. Situação na exportação, sem data", "usuarios"),
 ("relogio_sincronizou_90d", "uso", "Teve treino vindo do relógio na janela", "atividades"),
 ("pct_treinos_relogio", "uso", "% dos treinos da janela que vieram do relógio (vazio se não treinou)", "atividades"),
 ("dias_historico_posts", "uso", "Dias da janela de 90 dias com dados de posts (os posts começam em 01/07/2025). 0 = colunas de posts vazias", "derivada"),
 ("posts_90d", "uso", "Posts no feed na janela", "feed_posts"),
 ("postou_objetivo_cumprido", "uso", "Postou na janela um texto de objetivo cumprido (palavras-chave; aproximado)", "feed_posts"),
 ("postou_despedida", "uso", "Postou na janela um texto de despedida do app (palavras-chave; aproximado)", "feed_posts"),
 ("n_seguidores", "uso", "Contas que seguem o usuário na data de referência", "amizades"),
 ("n_seguindo", "uso", "Contas que o usuário segue na data de referência", "amizades"),
 ("n_mutuas", "uso", "Relações em que os dois se seguem na data de referência", "amizades"),
 ("treinos_acompanhados_90d", "uso", "Treinos feitos com outra conta na janela", "atividades"),
 ("nivel_engajamento", "comportamento", "alto (treinou nos últimos 7 dias), médio (treinou no mês, não na última semana), baixo (30 a 44 dias sem treinar), possível churn (45+ dias). Vazio com menos de 45 dias de histórico", "atividades"),
 ("ultimo_treino", "comportamento", "Data do último treino até a referência (vazio se nenhum treino desde 01/01/2026)", "atividades"),
 ("dias_sem_treinar", "comportamento", "Dias do último treino até a referência (vazio se nenhum treino no arquivo)", "atividades"),
 ("sem_treino_ate_referencia", "comportamento", "Nenhum treino no arquivo até a data de referência", "atividades"),
 ("treinos_7d", "comportamento", "Treinos nos 7 dias até a referência", "atividades"),
 ("treinos_30d", "comportamento", "Treinos nos 30 dias até a referência", "atividades"),
 ("treinos_90d", "comportamento", "Treinos na janela de 90 dias (ou na parte dela com dados)", "atividades"),
 ("treinos_por_semana", "comportamento", "treinos_90d ÷ semanas com dados (mínimo de 28 dias de histórico)", "derivada"),
 ("dias_ativos_90d", "comportamento", "Dias distintos com treino na janela", "atividades"),
 ("minutos_90d", "comportamento", "Minutos de treino na janela (duração 0 conta como ausente)", "atividades"),
 ("tem_objetivo", "comportamento", "Respondeu o objetivo no onboarding com conteúdo", "onboarding"),
 ("objetivo_texto", "comportamento", "Texto original do objetivo", "onboarding"),
 ("objetivo_categoria", "comportamento", "Tema do objetivo por palavras-chave: saúde, peso e forma, companhia, bem-estar mental, performance, só o evento, outro (aproximado)", "onboarding"),
 ("objetivo_evento", "comportamento", "O objetivo cita um evento com data: prova esportiva, evento pessoal ou sem evento (aproximado)", "onboarding"),
 ("objetivo_com_data", "comportamento", "Objetivo ligado a um evento com data", "onboarding"),
 ("nivel_declarado", "comportamento", "iniciante, intermediario ou avancado, declarado no onboarding", "onboarding"),
 ("dias_por_semana_declarado", "comportamento", "Dias de treino por semana pretendidos no onboarding (1 a 7)", "onboarding"),
 ("aderencia_meta", "comportamento", "treinos_por_semana ÷ dias_por_semana_declarado (1 = cumpre a meta)", "derivada"),
 ("n_cobrancas_aprovadas", "pagamento", "Cobranças aprovadas desde 07/2024", "pagamentos"),
 ("receita_liquida", "pagamento", "Receita líquida de taxas e devoluções desde 07/2024 (R$)", "pagamentos"),
 ("teve_recusa", "pagamento", "Teve cobrança recusada", "pagamentos"),
 ("teve_devolucao", "pagamento", "Teve chargeback ou estorno", "pagamentos"),
 ("n_chamados", "suporte", "Chamados de suporte abertos até a data de referência", "tickets_suporte"),
 ("chamado_falha_gps", "suporte", "Abriu chamado sobre a falha de GPS até a referência", "tickets_suporte"),
 ("atingida_falha_gps", "suporte", "Teve treino sem GPS (04/05 a 28/06/2026) ou chamado sobre o tema. Só Android", "atividades + tickets_suporte"),
 ("grupo_campanha", "campanha", "Grupo na campanha de 10/08/2026: controle, conteudo, desafio ou cupom (vazio se não sorteado)", "campanha_ago26"),
]
ordem = [d[0] for d in DIC]
faltam = [c for c in ordem if c not in m.columns]; sobram = [c for c in m.columns if c not in ordem]
print("no dicionário e fora da tabela:", faltam, "| na tabela e fora do dicionário:", sobram)
assert not faltam and not sobram
master = m[ordem].sort_values("user_id").reset_index(drop=True)
for c in ["entrou_com_cupom", "teve_recusa", "teve_devolucao", "assinatura_antes_do_log"]:
    master[c] = master[c].astype("boolean")
for c in ["n_tipos", "treinos_7d", "treinos_30d", "treinos_90d", "dias_ativos_90d", "minutos_90d", "posts_90d", "treinos_acompanhados_90d", "dias_sem_treinar", "duracao_dias", "n_cobrancas_aprovadas", "dias_historico_treinos", "dias_historico_posts"] + ["treinos_" + t for t in TIPOS]:
    master[c] = master[c].astype("Int64")

dic = pd.DataFrame(DIC, columns=["coluna", "grupo", "descricao", "fonte"])
dic["tipo"] = [str(master[c].dtype) for c in dic.coluna]
dic["preenchidas"] = [int(master[c].notna().sum()) for c in dic.coluna]
dic["% preenchido"] = (dic.preenchidas / len(master) * 100).round(1)
dic["% preenchido (ativos)"] = [round(master.loc[master.ativa_no_corte, c].notna().mean() * 100, 1) for c in dic.coluna]
master.to_parquet(P / "tabela_master.parquet", index=False)
master.to_csv(S / "tabela_master.csv", index=False, encoding="utf-8-sig")
dic.to_csv(S / "tabela_master_dicionario.csv", index=False, encoding="utf-8-sig")
print("gravado:", master.shape, "| colunas por grupo:", dic.grupo.value_counts().to_dict())
print(dic[["coluna", "grupo", "tipo", "% preenchido", "% preenchido (ativos)"]].to_string())

no dicionário e fora da tabela: [] | na tabela e fora do dicionário: []


gravado: (63929, 80) | colunas por grupo: {'uso': 22, 'comportamento': 18, 'entrada': 13, 'saída': 9, 'perfil': 7, 'pagamento': 4, 'identificação': 3, 'suporte': 3, 'campanha': 1}
                       coluna          grupo            tipo  % preenchido  % preenchido (ativos)
0                     user_id  identificação          object         100.0                  100.0
1                    situacao  identificação          object         100.0                  100.0
2             data_referencia  identificação  datetime64[ns]         100.0                  100.0
3                        sexo         perfil          object         100.0                  100.0
4              ano_nascimento         perfil           Int64          94.0                   93.9
5              idade_no_corte         perfil           Int64          94.0                   93.9
6                 faixa_idade         perfil          object         100.0                  100.0
7                      cidade       

In [9]:
print("exemplo: 3 assinantes ativos e 3 ex-assinantes de 2026")
ex = pd.concat([master[master.situacao == "assinante ativo"].head(3), master[(master.situacao == "ex-assinante") & (master.data_referencia >= "2026-06-01")].head(3)])
print(ex.T.to_string())

exemplo: 3 assinantes ativos e 3 ex-assinantes de 2026
                                           0                                                             9                    12                   68                              179                  207
user_id                             u_0001651d                                                    u_00130058           u_0014b486           u_005000a0                      u_00baf43b           u_00dc9c86
situacao                       assinante ativo                                               assinante ativo      assinante ativo         ex-assinante                    ex-assinante         ex-assinante
data_referencia            2026-09-30 00:00:00                                           2026-09-30 00:00:00  2026-09-30 00:00:00  2026-07-12 00:00:00             2026-08-01 00:00:00  2026-08-28 00:00:00
sexo                                         M                                                             F                    M